In [ ]:
import shapefile
import geopandas as gpd
import numpy as np

from matplotlib import pyplot as plt
from matplotlib.cm import get_cmap

from shapely.geometry import shape as shapely_shape

import torch
import networkx as nx
from torch_geometric.utils import to_networkx

## Load and Plot input Shapefile

In [ ]:
PATH_SHP = './data/waterways_merged_reprojected.shp'

# Load as GeoDataFrame
gdf = gpd.read_file(PATH_SHP)
gdf = gdf.to_crs(epsg=3857)  # project to metric (optional, useful for distance ops)
print(gdf.shape)

In [ ]:
# Plot the shapefile
gdf.plot(figsize=(10, 10), edgecolor='#143642', linewidth=0.5)
plt.title("Rivers Regular Lines")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.grid(True)
plt.show()

## Distribution of Original Line Lengths

### Meter Distribution

In [ ]:
def plot_shape_meter_distribution(path):
    """
    Reads a shapefile in EPSG:3068 (Gauss-Krüger, meters)
    and plots the distribution of shape lengths.
    Only the first 64 unique length values are shown.
    """
    sf = shapefile.Reader(path)

    lengths = []
    num_points = []

    for s in sf.shapes():
        geom = shapely_shape(s.__geo_interface__)
        lengths.append(round(geom.length))  # round to nearest km
        num_points.append(len(s.points))

    # Count frequencies of each unique length
    unique_lengths, counts = np.unique(lengths, return_counts=True)

    # Take only the first 64 values
    unique_lengths = unique_lengths[:1000]
    counts = counts[:1000]

    # Plot
    plt.figure()
    bars = plt.bar(unique_lengths, counts, color='#143642')
    plt.yscale('log')
    plt.title('Distribution of Shape Lengths')
    plt.xlabel('Length of Shape (km)')
    plt.ylabel('Count')
    plt.grid(True, axis='y', linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()

In [ ]:
sf = shapefile.Reader(PATH_SHP)

lengths = []
num_points = []

for s in sf.shapes():
    geom = shapely_shape(s.__geo_interface__)
    lengths.append(round(geom.length/1000))  # round to nearest km
    num_points.append(len(s.points))
    #print("km: ", geom.length)
    #print("num: ", len(s.points))
    print(geom.length/len(s.points))

In [ ]:
plot_shape_meter_distribution(PATH_SHP)

### Number of Points Distribution

In [ ]:
def plot_shape_length_distribution(path):
    """
    Reads a shapefile and plots the distribution of shape lengths
    (number of points per shape) with value labels on each bar.
    """
    with shapefile.Reader(path) as sf:
        shapes = sf.shapes()
        lengths = [len(shape.points) for shape in shapes]

    # Count frequencies
    unique_lengths, counts = np.unique(lengths, return_counts=True)

    unique_lengths = unique_lengths[:500]
    counts = counts[:500]

    # Plot
    plt.figure()
    bars = plt.bar(unique_lengths, counts, color='#143642')
    plt.yscale('log')
    plt.title('Distribution of Shape Lengths')
    plt.xlabel('Number of Points per Shape')
    plt.ylabel('Count')
    plt.grid(True, axis='y', linestyle='--', alpha=0.5)

    # Add vertical lines
    plt.axvline(x=32, color='#EC9A29', linestyle='--', linewidth=1.5, label='points=32')
    plt.axvline(x=256, color='#A8201A', linestyle='--', linewidth=1.5, label='points=256')

    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
plot_shape_length_distribution(PATH_SHP)

## Final preprocessed lines

In [ ]:
def plot_lines(lines, lines_syn_1, lines_syn_2, start=0, end=100, figsize=(10, 6)):
    """
    Plots a number of 2D lines (shape: [N, num_points, 2])

    Args:
        lines (np.ndarray): Shape (N, num_points, 2)
        num_to_plot (int): How many lines to plot
        figsize (tuple): Size of the figure
    """
    plt.figure(figsize=figsize)

    for i in range(start,end):
        line = lines[i]
        line_s1 = lines_syn_1[i]
        line_s2 = lines_syn_2[i]

        x, y = line[:, 0], line[:, 1]
        x_s1, y_s1 = line_s1[:, 0], line_s1[:, 1]
        x_s2, y_s2 = line_s2[:, 0], line_s2[:, 1]

        plt.plot(x, y, color='#143642', label=f'Original Line - {i}')
        plt.plot(x_s1, y_s1, color='#EC9A29', label=f'Synthetic Line Close - {i}')
        plt.plot(x_s2, y_s2, color='#A8201A', label=f'Synthetic Line Far - {i}')

        plt.title(f'Interpolated Lines - Line number {i}')
        plt.xlabel('X')
        plt.ylabel('Y')
        plt.axis('equal')
        plt.grid(True)
        plt.legend()
        plt.show()

In [ ]:
river_lines = np.load('./data/real-world-river.npy')
river_lines_close = np.load('./data/real-world-river-displaced_close.npy')
river_lines_far = np.load('./data/real-world-river-displaced_far.npy')

In [ ]:
plot_lines(river_lines, river_lines_close, river_lines_far, start=0, end=50)

## Plot final graph

In [ ]:
def plot_graph(data):
    # Convert PyG Data -> NetworkX
    G = to_networkx(data, to_undirected=True)

    # Extract node positions (x,y from features)
    pos = {i: (float(data.x[i][1]), float(data.x[i][2])) for i in range(data.num_nodes)}

    # Node colors by line_id
    colors = [int(data.x[i][0].item()) for i in range(data.num_nodes)]

    plt.figure(figsize=(8, 6))
    nx.draw(G, pos,
            node_size=30,
            node_color=colors,
            cmap=plt.cm.Set1,
            edge_color="lightgray",
            alpha=0.8)
    plt.show()

In [ ]:
graphs_seq = torch.load('./data/final_dataset/graph/graphs_sequential.pt', weights_only=False)
graph = graphs_seq[2]   # first sequence graph
plot_graph(graph)